# Notebook 06: FastAPI + ngrok Relay for the Serverless Endpoint

**Module:** ITI113 Machine Learning & Operations

**Focus Area:** C - MLOps & Deployment

---

### What this notebook does

This notebook sets up a temporary **FastAPI relay** inside SageMaker Studio/JupyterLab and exposes it using **ngrok**.



Flow:

```text
External UI / browser / external server
        ↓ HTTPS ngrok URL
FastAPI relay running in Team40 SageMaker space
        ↓ boto3 using SageMaker execution role
SageMaker endpoint: iti113-team03-crypto-scam-detector
```

This avoids putting AWS Access Key / Secret Access Key in an external app. The relay uses the SageMaker execution role attached to the Studio Space.

> Use this only for temporary demo/testing. Stop ngrok and the FastAPI server after testing.


## 0. Check notebook AWS identity

In [21]:
import boto3
import json

sts = boto3.client("sts")
identity = sts.get_caller_identity()

print(json.dumps(identity, indent=2))

{
  "UserId": "AROAQUXQWQCIVKBRB7UJK:SageMaker",
  "Account": "044528205969",
  "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker",
  "ResponseMetadata": {
    "RequestId": "5854c9c9-8d4f-4812-a0cb-f3fbb90660bf",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "5854c9c9-8d4f-4812-a0cb-f3fbb90660bf",
      "x-amz-sts-extended-request-id": "MTphcC1zb3V0aGVhc3QtMTpTOjE3ODcxMzA2NjQwNTA6UjpFcGRQTVV0bA==",
      "content-type": "text/xml",
      "content-length": "461",
      "date": "Wed, 19 Aug 2026 09:11:04 GMT"
    },
    "RetryAttempts": 0
  }
}


## 1. Configuration

In [ ]:
REGION = "ap-southeast-1"
TEAM_ID = "team03"
PROJECT_NAME = "crypto-scam-detector"

# Matches the naming used in Notebook 03.
ENDPOINT_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"

# Baseline (untuned) endpoint deployed by Notebook 03 Section 7, for live champion-vs-baseline
# comparison in the demo. Leave this as-is even if Notebook 03 Section 7 hasn't been run yet —
# the relay's /predict/baseline route will just fail until that endpoint exists.
BASELINE_ENDPOINT_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}-baseline"

# Change this before sharing the ngrok URL with anyone (or before a demo/presentation).
# The Streamlit app must send this value in the x-api-key header.
RELAY_API_KEY = "team03-xxx"  # change me

print("Region:", REGION)
print("Champion endpoint:", ENDPOINT_NAME)
print("Baseline endpoint:", BASELINE_ENDPOINT_NAME)
print("Relay API key set:", bool(RELAY_API_KEY))


Region: ap-southeast-1
Champion endpoint: iti113-team03-crypto-scam-detector
Baseline endpoint: iti113-team03-crypto-scam-detector-baseline
Relay API key set: True


## 2. Check that the endpoint exists

In [23]:
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name=REGION)

try:
    endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint status:", endpoint["EndpointStatus"])
    print("Endpoint ARN:", endpoint["EndpointArn"])
except ClientError as e:
    print("Could not describe endpoint.")
    print(e.response["Error"]["Code"])
    print(e.response["Error"]["Message"])

Endpoint status: InService
Endpoint ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/iti113-team03-crypto-scam-detector


## 3. Test direct endpoint invocation from the notebook

In [24]:
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

sample_message = (
    "URGENT: Your wallet has been selected for a guaranteed 100% profit airdrop! "
    "Deposit 500 USDT to your wallet address within 1 hour to claim now. "
    "Contact us on Telegram immediately, don't miss out!"
)

sample_payload = {"text": sample_message}

try:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample_payload),
    )

    result = response["Body"].read().decode("utf-8")
    print("Raw endpoint response:")
    print(result)

except ClientError as e:
    print("Endpoint invocation failed.")
    print(e.response["Error"]["Code"])
    print(e.response["Error"]["Message"])

Raw endpoint response:
[{"prediction": 1, "label": "Scam", "probability": 0.9985, "top_features": [{"feature": "urgency_score", "contribution": 2.6441, "direction": "scam"}, {"feature": "your", "contribution": 1.5343, "direction": "scam"}, {"feature": "your wallet", "contribution": 0.6228, "direction": "scam"}, {"feature": "urgency_keyword_count", "contribution": -0.5157, "direction": "legitimate"}, {"feature": "countdown_phrase_count", "contribution": 0.4075, "direction": "scam"}]}]


## 4. Install FastAPI, Uvicorn, and pyngrok

Run this once in the notebook environment.

In [25]:
%pip install -q fastapi uvicorn pyngrok requests

Note: you may need to restart the kernel to use updated packages.


## 5. Create the FastAPI relay app

In [26]:
%%writefile relay_api.py

from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
import boto3
import json
import os

REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "iti113-team03-crypto-scam-detector")
BASELINE_ENDPOINT_NAME = os.environ.get(
    "BASELINE_ENDPOINT_NAME", "iti113-team03-crypto-scam-detector-baseline"
)
RELAY_API_KEY = os.environ.get("RELAY_API_KEY", "team03-demo-key-change-me")

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

app = FastAPI(title="Team03 Crypto Scam Detector Relay")

# For temporary demo use. For production, restrict allowed_origins.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def home():
    return {
        "status": "running",
        "service": "Team03 FastAPI relay",
        "champion_endpoint": ENDPOINT_NAME,
        "baseline_endpoint": BASELINE_ENDPOINT_NAME,
        "routes": ["GET /health", "POST /predict", "POST /predict/baseline", "GET /docs"],
    }


@app.get("/health")
def health():
    return {"status": "ok"}


def _invoke(endpoint_name: str, x_api_key: str, request_json):
    if x_api_key != RELAY_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(request_json),
        )

        result = response["Body"].read().decode("utf-8")

        try:
            return json.loads(result)
        except Exception:
            return {"raw_result": result}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict")
async def predict(request: Request, x_api_key: str = Header(None)):
    # Keep the relay narrow: call only the fixed champion endpoint configured on the
    # server side. The payload (text / instances) is forwarded as-is; inference.py
    # handles both shapes.
    try:
        payload = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Request body must be valid JSON")
    return _invoke(ENDPOINT_NAME, x_api_key, payload)


@app.post("/predict/baseline")
async def predict_baseline(request: Request, x_api_key: str = Header(None)):
    # Same narrow behaviour as /predict, but calls the baseline (untuned) endpoint
    # deployed by Notebook 03 Section 7, for live champion-vs-baseline comparison in the demo.
    try:
        payload = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Request body must be valid JSON")
    return _invoke(BASELINE_ENDPOINT_NAME, x_api_key, payload)


Overwriting relay_api.py


## 6. Start FastAPI server from the notebook

Starts Uvicorn in the background on port `8000`. If you prefer, run the same command in a SageMaker Terminal:

```bash
uvicorn relay_api:app --host 0.0.0.0 --port 8000
```

In [27]:
import os
import subprocess
import time

# Pass config to relay_api.py through environment variables
os.environ["AWS_REGION"] = REGION
os.environ["ENDPOINT_NAME"] = ENDPOINT_NAME
os.environ["BASELINE_ENDPOINT_NAME"] = BASELINE_ENDPOINT_NAME
os.environ["RELAY_API_KEY"] = RELAY_API_KEY

# Stop previous server process if this cell was run before
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("Stopped previous FastAPI server.")
except Exception:
    pass

relay_process = subprocess.Popen(
    ["uvicorn", "relay_api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)
print("FastAPI server process started.")
print("PID:", relay_process.pid)
print("Local URL: http://127.0.0.1:8000")
print("Docs URL:  http://127.0.0.1:8000/docs")


Stopped previous FastAPI server.
FastAPI server process started.
PID: 5519
Local URL: http://127.0.0.1:8000
Docs URL:  http://127.0.0.1:8000/docs


## 7. Test the local FastAPI relay

Tests the relay locally before exposing it with ngrok.

In [28]:
import requests

local_url = "http://127.0.0.1:8000/predict"

headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json",
}

response = requests.post(local_url, headers=headers, json=sample_payload, timeout=60)

print("Status code:", response.status_code)
print("Response text:")
print(response.text)

try:
    print("Parsed JSON:")
    print(json.dumps(response.json(), indent=2))
except Exception:
    pass

Status code: 200
Response text:
[{"prediction":1,"label":"Scam","probability":0.9985,"top_features":[{"feature":"urgency_score","contribution":2.6441,"direction":"scam"},{"feature":"your","contribution":1.5343,"direction":"scam"},{"feature":"your wallet","contribution":0.6228,"direction":"scam"},{"feature":"urgency_keyword_count","contribution":-0.5157,"direction":"legitimate"},{"feature":"countdown_phrase_count","contribution":0.4075,"direction":"scam"}]}]
Parsed JSON:
[
  {
    "prediction": 1,
    "label": "Scam",
    "probability": 0.9985,
    "top_features": [
      {
        "feature": "urgency_score",
        "contribution": 2.6441,
        "direction": "scam"
      },
      {
        "feature": "your",
        "contribution": 1.5343,
        "direction": "scam"
      },
      {
        "feature": "your wallet",
        "contribution": 0.6228,
        "direction": "scam"
      },
      {
        "feature": "urgency_keyword_count",
        "contribution": -0.5157,
        "direct

## 8. Configure ngrok

You need a free ngrok account and auth token.

1. Go to the ngrok dashboard.
2. Copy your auth token.
3. Paste it below.

Do **not** commit the token to GitHub.

In [29]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste ngrok auth token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok auth token configured for this session.")

Paste ngrok auth token:  ········


ngrok auth token configured for this session.


## 9. Start ngrok tunnel

Creates a temporary public HTTPS URL forwarding to local port `8000`.

In [33]:
from pyngrok import ngrok

# Close old tunnels from this notebook session
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception:
    pass

public_tunnel = ngrok.connect(8000, "http")
public_url = public_tunnel.public_url

print("Public ngrok URL:")
print(public_url)
print("FastAPI docs:")
print(public_url + "/docs")
print("Champion prediction route (paste this into the Streamlit sidebar):")
print(public_url + "/predict")
print("Baseline prediction route (Streamlit derives this automatically):")
print(public_url + "/predict/baseline")


t=2026-08-19T09:13:27+0000 lvl=warn msg="Stopping forwarder" name=http-8000-841fdd39-fccc-417e-9ba4-38f77554f5b9 acceptErr="failed to accept connection: Listener closed"


Public ngrok URL:
https://boaster-prodigy-silica.ngrok-free.dev
FastAPI docs:
https://boaster-prodigy-silica.ngrok-free.dev/docs
Champion prediction route (paste this into the Streamlit sidebar):
https://boaster-prodigy-silica.ngrok-free.dev/predict
Baseline prediction route (Streamlit derives this automatically):
https://boaster-prodigy-silica.ngrok-free.dev/predict/baseline


## 10. Test the ngrok public URL from the notebook

Simulates the Streamlit app calling the public ngrok URL.

In [31]:
response = requests.post(public_url + "/predict", headers=headers, json=sample_payload, timeout=60)

print("Status code:", response.status_code)
print("Response text:")
print(response.text)

Status code: 200
Response text:
[{"prediction":1,"label":"Scam","probability":0.9985,"top_features":[{"feature":"urgency_score","contribution":2.6441,"direction":"scam"},{"feature":"your","contribution":1.5343,"direction":"scam"},{"feature":"your wallet","contribution":0.6228,"direction":"scam"},{"feature":"urgency_keyword_count","contribution":-0.5157,"direction":"legitimate"},{"feature":"countdown_phrase_count","contribution":0.4075,"direction":"scam"}]}]


## 11. Wire this into Streamlit

`pages/1_Scam_Detector.py` reads two values from its sidebar: the relay predict URL and the relay API key. Paste in:

- Relay Predict URL: the `public_url + "/predict"` printed above (the champion route)
- Relay API Key: the `RELAY_API_KEY` set in Section 1

The app sends `{"text": message}` to that URL with the `x-api-key` header for the champion prediction, exactly as tested in Section 10 — and automatically derives `.../predict/baseline` from the same base URL to also show the baseline model's prediction alongside it, once Notebook 03 Section 7's baseline endpoint has been deployed.


## 12. Stop ngrok and FastAPI after the demo

Run this when testing/presentation is complete, to avoid unexpected endpoint invocations and cost.

In [32]:
# # Stop ngrok tunnels
# try:
#     ngrok.kill()
#     print("ngrok tunnels stopped.")
# except Exception as e:
#     print("ngrok stop issue:", e)

# # Stop FastAPI server
# try:
#     relay_process.terminate()
#     relay_process.wait(timeout=5)
#     print("FastAPI server stopped.")
# except Exception as e:
#     print("FastAPI stop issue:", e)

## Troubleshooting

### `AccessDeniedException` from SageMaker Runtime

The team03 execution role needs permission to invoke the endpoint:

```json
{
  "Effect": "Allow",
  "Action": "sagemaker:InvokeEndpoint",
  "Resource": "arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/iti113-team03-crypto-scam-detector"
}
```

### `404` or endpoint not found

Check that the endpoint name is exactly:

```text
iti113-team03-crypto-scam-detector
```

### Streamlit gets `401 Invalid API key`

Make sure the Streamlit sidebar's Relay API Key field matches `RELAY_API_KEY` from Section 1 exactly.

### ngrok URL changes

Free ngrok URLs usually change every time the tunnel is restarted. Re-paste the new `/predict` URL into the Streamlit sidebar after restarting this notebook.

### Security note

This relay should expose only fixed routes such as `/predict`. Do not create routes that accept arbitrary Python code, arbitrary AWS actions, S3 paths, or arbitrary SageMaker endpoint names.